In [ ]:
# =========================================================================
# Master Benchmarking Notebook for Spatio-Temporal Coastal SLA Forecasting
# =========================================================================

# =========================================================================
# SECTION 1: CENTRALIZED CONFIGURATION & REPRODUCIBILITY LOCK 
# =========================================================================
import os
import gc
import time
import random
import json
import joblib
import numpy as np
import pandas as pd
from datetime import datetime

RANDOM_SEED = 42
BATCH_SIZE = 16
EPOCHS = 100
LEARNING_RATE = 5e-4  
EARLY_STOPPING_PATIENCE = 10
EXTREME_WEIGHT = 3.0

os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import ConvLSTM2D, BatchNormalization as BatchNorm, Conv2D, Dense, Dropout, LSTM, Reshape, Lambda
from tensorflow.keras.callbacks import EarlyStopping, CSVLogger

from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor

tf.random.set_seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)
tf.config.experimental.enable_op_determinism()

# DIRECTORY ROOT ASSIGNMENT 
INPUT_ROOT  = "/kaggle/input/datasets/muhffikkri/sea-level-anomaly-semarang"
COASTAL_POINT_FILE = os.path.join(INPUT_ROOT, "reference", "coastal_points.parquet")
SCENARIO_INPUT_DIR = os.path.join(INPUT_ROOT, "scenarios")
EDA_SUMMARY_FILE   = os.path.join(INPUT_ROOT, "metadata", "eda_summary.json")

OUTPUT_ROOT  = "/kaggle/working"
MODEL_OUTPUT_DIR    = os.path.join(OUTPUT_ROOT, "models")
PRED_OUTPUT_DIR     = os.path.join(OUTPUT_ROOT, "predictions")
METADATA_OUTPUT_DIR = os.path.join(OUTPUT_ROOT, "metadata")
HISTORY_OUTPUT_DIR  = os.path.join(OUTPUT_ROOT, "history")

TRIAL_SCENARIOS = [
    # EXP1_ULTRA_6030 
    (60,30),
    
    # EXP2_HEAVY_MIX_A
    (60,15),
    (60,1),
    (30,15),
    (30,7),
    (15,7),
    (15,3),
    
    # EXP3_HEAVY_MIX_B
    (60,7),
    (60,3),
    (30,30),
    (30,3),
    (30,1),
    (15,1),
    (7,3),
    (7,1),
    (3,1)
]

for folder in [MODEL_OUTPUT_DIR, PRED_OUTPUT_DIR, METADATA_OUTPUT_DIR, HISTORY_OUTPUT_DIR]:
    os.makedirs(folder, exist_ok=True)

# VRAM Allocation Optimization for Tesla T4
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"[+] GPU Active: {gpus}. Strict Paper-Level Determinism Locked.")
    except RuntimeError as e:
        print(f"[-] GPU Warning: {e}")


# =========================================================================
# SECTION 2: RIGOROUS LOSS & SCIENTIFIC EVALUATION ENGINES
# =========================================================================
def get_coastal_mask(grid_h, grid_w):
    coastal_df = pd.read_parquet(COASTAL_POINT_FILE)
    mask = np.zeros((grid_h, grid_w), dtype=np.float32)
    for _, row in coastal_df.iterrows():
        lat_idx, lon_idx = int(row['lat_idx']), int(row['lon_idx'])
        if lat_idx < grid_h and lon_idx < grid_w:
            mask[lat_idx, lon_idx] = 1.0
    return mask

def masked_coastal_mse(coastal_mask_tf):
    def loss(y_true, y_pred):
        y_true_coastal = y_true * coastal_mask_tf
        y_pred_coastal = y_pred * coastal_mask_tf
        squared_errors = tf.square(y_true_coastal - y_pred_coastal)
        
        total_error = tf.reduce_sum(squared_errors)
        normalizer = (
            tf.cast(tf.shape(y_true)[0], tf.float32) * tf.cast(tf.shape(y_true)[1], tf.float32) * tf.reduce_sum(coastal_mask_tf)
        )
        return total_error / (normalizer + 1e-7)
    return loss

def weighted_extreme_mse(extreme_threshold_scaled, coastal_mask_tf):
    def loss(y_true, y_pred):
        y_true_coastal = y_true * coastal_mask_tf
        y_pred_coastal = y_pred * coastal_mask_tf
        squared_errors = tf.square(y_true_coastal - y_pred_coastal)
        extreme_mask = tf.cast(
            y_true_coastal >= extreme_threshold_scaled,
            tf.float32
        )
        weight_matrix = 1.0 + (
            extreme_mask * (EXTREME_WEIGHT - 1.0)
        )
        weighted_error = tf.reduce_sum(
            squared_errors * weight_matrix
        )
        normalizer = tf.reduce_sum(weight_matrix)
        return weighted_error / (normalizer + 1e-7)
    return loss

def calculate_inverse_metrics(y_true_m, y_pred_m, extreme_p95_physical, coastal_mask_arr, persistence_rmse_physical=None):
    mask_idx = (coastal_mask_arr == 1.0)
    y_true_c = y_true_m[:, :, mask_idx]
    y_pred_c = y_pred_m[:, :, mask_idx]
    
    mse = np.nanmean((y_true_c - y_pred_c) ** 2)
    rmse = np.sqrt(mse)
    mae = np.nanmean(np.abs(y_true_c - y_pred_c))
    
    extreme_mask = (y_true_c >= extreme_p95_physical)
    if np.sum(extreme_mask) > 0:
        ext_mse = np.nanmean((y_true_c[extreme_mask] - y_pred_c[extreme_mask]) ** 2)
        ext_rmse = np.sqrt(ext_mse)
        ext_mae = np.nanmean(np.abs(y_true_c[extreme_mask] - y_pred_c[extreme_mask]))
    else:
        ext_rmse, ext_mae = 0.0, 0.0
        
    pss = 0.0
    if persistence_rmse_physical is not None and persistence_rmse_physical > 0:
        pss = 1.0 - (rmse / persistence_rmse_physical)
        
    return {
        "RMSE_m": float(rmse), 
        "MAE_m": float(mae), 
        "MSE_m2": float(mse), 
        "Extreme_RMSE_m": float(ext_rmse), 
        "Extreme_MAE_m": float(ext_mae), 
        "PSS": float(pss)
    }


# =========================================================================
# SECTION 3: MINIMALIST PIXEL-WISE CONVOLUTION ARCHITECTURES
# =========================================================================
def convlstm2d_model(input_shape, target_shape, grid_h, grid_w, loss_fn):
    model = Sequential([
        tf.keras.Input(shape=input_shape),
        ConvLSTM2D(filters=8, kernel_size=(1,1), padding='same', return_sequences=True, activation='tanh', recurrent_activation='sigmoid'),
        Dropout(0.1),
        ConvLSTM2D(filters=4, kernel_size=(1,1), padding='same', return_sequences=False, activation='tanh', recurrent_activation='sigmoid'),
        Conv2D(filters=target_shape[0], kernel_size=(1,1), activation='linear', padding='same'),
        Lambda(lambda x:tf.expand_dims(tf.transpose(x, [0,3,1,2]), -1))
    ])

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=1.0), loss=loss_fn)
    assert model.output_shape == (None, target_shape[0], grid_h, grid_w, 1), f"Architecture shape mismatch: {model.output_shape}"
    return model

def lstm_model(input_window, n_coastal_cells, output_horizon):
    model = Sequential([
        tf.keras.Input(shape=(input_window, n_coastal_cells)),
        LSTM(128, return_sequences=True),
        Dropout(0.2),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(output_horizon * n_coastal_cells, activation='linear'),
        Reshape((output_horizon, n_coastal_cells))
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss='mse')
    return model


# =========================================================================
# SECTION 4: MASTER FORECASTING EXPERIMENT PIPELINE ENGINE
# =========================================================================
def train_and_evaluate_scenario(input_window, output_horizon):
    tf.keras.backend.clear_session()
    gc.collect()
    
    scenario_name = f"scenario_{input_window}_{output_horizon}"
    scenario_dir = os.path.join(SCENARIO_INPUT_DIR, scenario_name)
    
    print("\n" + "="*75)
    print(f"EXPERIMENT RUNTIME ACTIVE FOR: {scenario_name.upper()}")
    print("="*75)
    
    data_bundle = np.load(os.path.join(scenario_dir, f"{scenario_name}_compressed.npz"))
    X_train_raw, y_train_raw = data_bundle['X_train'], data_bundle['y_train']
    X_val_raw, y_val_raw     = data_bundle['X_val'], data_bundle['y_val']
    X_test_raw, y_test_raw   = data_bundle['X_test'], data_bundle['y_test']
    
    grid_h, grid_w = X_train_raw.shape[2], X_train_raw.shape[3]
    coastal_mask_arr = get_coastal_mask(grid_h, grid_w)
    coastal_flat_idx = (coastal_mask_arr == 1.0)
    coastal_mask_tf  = tf.reshape(tf.convert_to_tensor(coastal_mask_arr, dtype=tf.float32), (1, 1, grid_h, grid_w, 1))
    
    def compile_2channel_tensor(raw_x_tensor):
        ocean_mask = (~np.isnan(raw_x_tensor)).astype(np.float32)
        sla_cleaned = np.nan_to_num(raw_x_tensor, nan=0.0)
        return np.concatenate([sla_cleaned, ocean_mask], axis=-1)
        
    X_train = compile_2channel_tensor(X_train_raw)
    X_val   = compile_2channel_tensor(X_val_raw)
    X_test  = compile_2channel_tensor(X_test_raw)
    
    y_train = np.nan_to_num(y_train_raw, nan=0.0)
    y_val   = np.nan_to_num(y_val_raw, nan=0.0)
    y_test  = np.nan_to_num(y_test_raw, nan=0.0) 
    
    scaler = joblib.load(os.path.join(scenario_dir, "scaler.pkl"))
    with open(EDA_SUMMARY_FILE, "r") as f:
        eda_meta = json.load(f)
    extreme_p95_physical = eda_meta["extreme_threshold_meters"]
    extreme_threshold_scaled = float(scaler.transform([[extreme_p95_physical]])[0][0])
    
    results_bundle = {}
    time_log_bundle = {}
    param_log_bundle = {}
    history_summary = {}
    
    callbacks = [EarlyStopping(monitor='val_loss', mode='min', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True)]

    y_true_m = scaler.inverse_transform(y_test.reshape(-1, 1).astype(np.float32)).reshape(y_test.shape)[..., 0]
    y_true_m[:, :, ~coastal_flat_idx] = np.nan

    # BASELINE A: OPERATIONAL PERSISTENCE FORECAST 
    print("\n[+] Calculating Operational Persistence Baseline (Vectorized)...")
    start_t = time.perf_counter()
    
    last_input = X_test_raw[:, -1:, :, :, :]
    y_pred_persistence_scaled = np.repeat(last_input, output_horizon, axis=1) 
            
    y_pred_persistence_m = scaler.inverse_transform(y_pred_persistence_scaled.reshape(-1, 1).astype(np.float32)).reshape(y_pred_persistence_scaled.shape)[..., 0]
    y_pred_persistence_m[:, :, ~coastal_flat_idx] = np.nan
    
    time_log_bundle["Persistence_inference_seconds"] = time.perf_counter() - start_t
    param_log_bundle["Persistence_total_parameters"] = 0
    
    persistence_metrics = calculate_inverse_metrics(y_true_m, y_pred_persistence_m, extreme_p95_physical, coastal_mask_arr)
    results_bundle["Persistence"] = persistence_metrics
    persistence_rmse_physical = persistence_metrics["RMSE_m"]
    print(f"    -> Operational Persistence Skill: RMSE = {persistence_rmse_physical:.4f} m")

    # BASELINE B: OPTIMIZED TREE-BASED MACHINE LEARNING (XGBOOST) 
    print("\n[+] Training Tabular Multi-Output XGBoost Regressor (Wrapper Parallelism)...")
    X_train_xgb = X_train_raw[..., 0][:, :, coastal_flat_idx].reshape(len(X_train), -1)
    X_test_xgb  = X_test_raw[..., 0][:, :, coastal_flat_idx].reshape(len(X_test), -1)
    y_train_xgb = y_train[..., 0][:, :, coastal_flat_idx].reshape(len(y_train), -1)
    
    X_train_xgb = np.nan_to_num(X_train_xgb, nan=0.0)
    X_test_xgb  = np.nan_to_num(X_test_xgb, nan=0.0)

    xgb_core = XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.05, 
        subsample=0.8, colsample_bytree=0.8, tree_method='hist', n_jobs=1, random_state=RANDOM_SEED
    )
    xgb_multi = MultiOutputRegressor(xgb_core, n_jobs=-1)
    
    start_t = time.perf_counter()
    xgb_multi.fit(X_train_xgb, y_train_xgb)
    time_log_bundle["XGBoost_train_seconds"] = time.perf_counter() - start_t
    
    start_t = time.perf_counter()
    y_pred_xgb_coastal = xgb_multi.predict(X_test_xgb).reshape(len(X_test), output_horizon, -1)
    time_log_bundle["XGBoost_inference_seconds"] = time.perf_counter() - start_t
    param_log_bundle["XGBoost_total_trees"] = sum(est.get_booster().num_boosted_rounds() for est in xgb_multi.estimators_)
    
    y_pred_xgb_scaled = np.zeros_like(y_test, dtype=np.float32)
    for s in range(len(y_test)):
        for h in range(output_horizon):
            y_pred_xgb_scaled[s, h, :, :, 0][coastal_flat_idx] = y_pred_xgb_coastal[s, h, :]
            
    y_pred_xgb_m = scaler.inverse_transform(y_pred_xgb_scaled.reshape(-1, 1).astype(np.float32)).reshape(y_pred_xgb_scaled.shape)[..., 0]
    y_pred_xgb_m[:, :, ~coastal_flat_idx] = np.nan
    
    xgb_metrics = calculate_inverse_metrics(y_true_m, y_pred_xgb_m, extreme_p95_physical, coastal_mask_arr, persistence_rmse_physical)
    results_bundle["XGBoost"] = xgb_metrics
    print(f"    -> Classical XGBoost Skill: RMSE = {xgb_metrics['RMSE_m']:.4f} m")

    # BASELINE C: TEMPORAL-ONLY DEEP LEARNING (STACKED LSTM) 
    print("\n[+] Training High-Capacity Stacked Sequential 1D LSTM...")
    n_coastal_cells = int(np.sum(coastal_flat_idx))
    X_train_lstm = X_train_raw[..., 0][:, :, coastal_flat_idx].reshape(X_train.shape[0], X_train.shape[1], -1)
    X_val_lstm   = X_val_raw[..., 0][:, :, coastal_flat_idx].reshape(X_val.shape[0], X_val.shape[1], -1)
    X_test_lstm  = X_test_raw[..., 0][:, :, coastal_flat_idx].reshape(X_test.shape[0], X_test.shape[1], -1)
    
    X_train_lstm = np.nan_to_num(X_train_lstm, nan=0.0)
    X_val_lstm   = np.nan_to_num(X_val_lstm, nan=0.0)
    X_test_lstm  = np.nan_to_num(X_test_lstm, nan=0.0)

    y_train_lstm = y_train[..., 0][:, :, coastal_flat_idx].reshape(len(y_train), output_horizon, -1)
    y_val_lstm   = y_val[..., 0][:, :, coastal_flat_idx].reshape(len(y_val), output_horizon, -1)
    
    lstm_model = lstm_model(input_window, n_coastal_cells, output_horizon)
    lstm_logger = CSVLogger(os.path.join(HISTORY_OUTPUT_DIR, f"{scenario_name}_lstm_history.csv"))
    param_log_bundle["LSTM_trainable_parameters"] = int(lstm_model.count_params())
    
    start_t = time.perf_counter()
    h_lstm = lstm_model.fit(X_train_lstm, y_train_lstm, validation_data=(X_val_lstm, y_val_lstm), 
                   batch_size=BATCH_SIZE, epochs=EPOCHS, callbacks=callbacks + [lstm_logger], verbose=0)
    time_log_bundle["LSTM_train_seconds"] = time.perf_counter() - start_t
    
    history_summary["LSTM_best_epoch"] = int(np.argmin(h_lstm.history['val_loss']) + 1)
    history_summary["LSTM_min_val_loss"] = float(np.min(h_lstm.history['val_loss']))

    start_t = time.perf_counter()
    y_pred_lstm_coastal = lstm_model.predict(X_test_lstm, batch_size=BATCH_SIZE, verbose=0)
    time_log_bundle["LSTM_inference_seconds"] = time.perf_counter() - start_t
    
    y_pred_lstm_scaled = np.zeros_like(y_test, dtype=np.float32)
    for s in range(len(y_test)):
        for h in range(output_horizon):
            y_pred_lstm_scaled[s, h, :, :, 0][coastal_flat_idx] = y_pred_lstm_coastal[s, h, :]
            
    y_pred_lstm_m = scaler.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1).astype(np.float32)).reshape(y_pred_lstm_scaled.shape)[..., 0]
    y_pred_lstm_m[:, :, ~coastal_flat_idx] = np.nan
    
    lstm_metrics = calculate_inverse_metrics(y_true_m, y_pred_lstm_m, extreme_p95_physical, coastal_mask_arr, persistence_rmse_physical)
    results_bundle["LSTM"] = lstm_metrics
    print(f"    -> Stacked LSTM Deep Learning Skill: RMSE = {lstm_metrics['RMSE_m']:.4f} m")

    # PROPOSED MODEL: WEIGHTED EXTREME-AWARE CONVLSTM2D 
    print("\n[+] Training Custom Weighted Extreme-Aware ConvLSTM2D Framework...")
    loss_fn_weighted = weighted_extreme_mse(extreme_threshold_scaled, coastal_mask_tf)
    convlstm_w = convlstm2d_model(X_train.shape[1:], y_train.shape[1:], grid_h, grid_w, loss_fn=loss_fn_weighted)
    conv_w_logger = CSVLogger(os.path.join(HISTORY_OUTPUT_DIR, f"{scenario_name}_convlstm_weighted_history.csv"))
    param_log_bundle["Weighted_ConvLSTM_trainable_parameters"] = int(convlstm_w.count_params())
    
    start_t = time.perf_counter()
    h_w = convlstm_w.fit(X_train, y_train, validation_data=(X_val, y_val), 
                   batch_size=BATCH_SIZE, epochs=EPOCHS, callbacks=callbacks + [conv_w_logger], verbose=0)
    time_log_bundle["Weighted_ConvLSTM_train_seconds"] = time.perf_counter() - start_t
    
    history_summary["Weighted_ConvLSTM_best_epoch"] = int(np.argmin(h_w.history['val_loss']) + 1)
    history_summary["Weighted_ConvLSTM_min_val_loss"] = float(np.min(h_w.history['val_loss']))
    convlstm_w.save(os.path.join(MODEL_OUTPUT_DIR, f"{scenario_name}_convlstm_weighted.keras"))
    
    start_t = time.perf_counter()
    y_pred_w_scaled = convlstm_w.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
    time_log_bundle["Weighted_ConvLSTM_inference_seconds"] = time.perf_counter() - start_t
    
    y_pred_w_m = scaler.inverse_transform(y_pred_w_scaled.reshape(-1, 1).astype(np.float32)).reshape(y_pred_w_scaled.shape)[..., 0]
    y_pred_w_m[:, :, ~coastal_flat_idx] = np.nan
    
    w_conv_metrics = calculate_inverse_metrics(y_true_m, y_pred_w_m, extreme_p95_physical, coastal_mask_arr, persistence_rmse_physical)
    results_bundle["Weighted_ConvLSTM"] = w_conv_metrics
    print(f"    -> Proposed Weighted ConvLSTM Skill: RMSE = {w_conv_metrics['RMSE_m']:.4f} m")

    # ARCHIVE RUNTIME METADATA & COMPRESSED TENSORS 
    runtime_summary = {
        "evaluation_metrics": results_bundle,
        "computational_times_seconds": time_log_bundle,
        "trainable_components_count": param_log_bundle,
        "convergence_statistics": history_summary
    }
    
    with open(os.path.join(METADATA_OUTPUT_DIR, f"{scenario_name}_evaluation_meters.json"), "w") as f:
        json.dump(runtime_summary, f, indent=4)
        
    np.savez_compressed(os.path.join(PRED_OUTPUT_DIR, f"{scenario_name}_predictions_physical_meters.npz"),
                        ground_truth_m=y_true_m, 
                        pred_persistence_m=y_pred_persistence_m,
                        pred_xgb_m=y_pred_xgb_m, pred_lstm_m=y_pred_lstm_m,
                        pred_weighted_convlstm_m=y_pred_w_m)

    print(f"Skenario {scenario_name} selesai. File fisis diekspor murni ke /kaggle/working.")
    
    del X_train, y_train, X_val, y_val, X_test, y_test, y_true_m
    del lstm_model, convlstm_std, convlstm_w, xgb_multi
    tf.keras.backend.clear_session()
    gc.collect()
    
    return {
        "Persistence": results_bundle["Persistence"]["RMSE_m"],
        "XGBoost": results_bundle["XGBoost"]["RMSE_m"],
        "LSTM": results_bundle["LSTM"]["RMSE_m"],
        "ConvLSTM": results_bundle["ConvLSTM"]["RMSE_m"],
        "Weighted_ConvLSTM": results_bundle["Weighted_ConvLSTM"]["RMSE_m"]
    }


# =========================================================================
# SECTION 5: MASTER BENCHMARK RUNNER & CSV AGGREGATOR LOOP
# =========================================================================
if __name__ == "__main__":
    csv_rows_accumulator = []
    
    for in_w, out_h in TRIAL_SCENARIOS:
        start_time = datetime.now()
        
        rmse_summary_dict = train_and_evaluate_scenario(input_window=in_w, output_horizon=out_h)
        
        row_entry = {
            "Input_Window_T": in_w,
            "Output_Horizon_H": out_h,
            "Persistence_RMSE_m": rmse_summary_dict["Persistence"],
            "XGBoost_RMSE_m": rmse_summary_dict["XGBoost"],
            "LSTM_RMSE_m": rmse_summary_dict["LSTM"],
            "ConvLSTM_RMSE_m": rmse_summary_dict["ConvLSTM"],
            "Weighted_ConvLSTM_RMSE_m": rmse_summary_dict["Weighted_ConvLSTM"]
        }
        csv_rows_accumulator.append(row_entry)
        
        print(f"[!] Compilation Time for Skenario ({in_w}_{out_h}): {datetime.now() - start_time}")
        print("-" * 75)
        
    print("\n" + "=" * 50)
    print("[+] Compiling Master Tabular Recap to CSV Summary File...")
    print("=" * 50)
    
    df_recap_summary = pd.DataFrame(csv_rows_accumulator)
    csv_summary_path = os.path.join(OUTPUT_ROOT, "rmse_models_summary.csv")
    df_recap_summary.to_csv(csv_summary_path, index=False)
    
    print(f"SUCCESS: Master Metrics Table exported safely to:")
    print(f"   -> {csv_summary_path}")
    print("\nPreview Rekapitulasi Tabel Metrik RMSE (Meter):")
    print(df_recap_summary.to_string(index=False))
    print("=" * 75)

[+] GPU Hardware Initialized: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]. Memory growth enabled.
✓ Section 1: Centralized Configuration Block Successfully Locked

KAGGLE HIGH-CAPACITY ENGINE OPERATIONAL FOR: SCENARIO_15_7


I0000 00:00:1783282044.112992      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783282044.115792      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(



[+] Calculating Operational Persistence Baseline...
    -> Operational Persistence Skill: RMSE = 0.0244 m

[+] Training Tabular Multi-Output XGBoost Regressor (Coastal Nodes Isolated)...
